#### LIBRARY IMPORTS

In [1]:
import pandas as pd
import numpy as np
import glob # For file pattern matching for loading files
import os # Certain debugging operations
import joblib # To save scaler and encoder

from pathlib import Path
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report,
)

import joblib
import copy # for deepcopy in save_results()

# Neural Netowrk specifc imports
from sklearn.linear_model import LogisticRegression


#### CONFIGURATOINS

In [2]:
FEATURE_ROOT = "eGeMAPs_features"
OUTPUT_ROOT = "model_outputs/logistic_regression"
RANDOM_SEED = 42
TRAIN_RATIO = 0.8
VAL_RATIO = 0.1
TEST_RATIO = 0.1

THRESHOLD = 0.5

LR_PENALTY = "l2"
LR_C = 1.0
LR_SOLVER = "liblinear"
LR_MAX_ITER = 1000

#### FUNCTIONS

In [3]:
def load_participant_data(participant_folder):
    csv_files = sorted(glob.glob(f"{FEATURE_ROOT}/{participant_folder}/*.csv")) # For multiple files

    # print(df.head())
    # print(df.shape)

    # Train / Validation / Test Split
    
    print(f"{participant_folder}: {len(csv_files)} CSV files")
    
    # # 80% train, 20% temp

    # Read every session and combine them into one dataframe
    full_df = pd.concat(
        [pd.read_csv(f) for f in csv_files],
        ignore_index=True
    )   
    print(f"Total samples: {len(full_df)}")
    
    # Remove unnecessary columns
    drop_cols = ["Unnamed: 0", "start_time", "end_time"]
    full_df = full_df.drop(
        columns=[c for c in drop_cols if c in full_df.columns]
    )
    print(full_df.columns)

    # Seperate Features and Labels
    X = full_df.drop(columns=["label"])
    y = full_df["label"]

    # Return ALL samples for this participant.
    return X.reset_index(drop=True), y.reset_index(drop=True)

In [4]:
def preprocess_data(X_train, X_test, y_train, y_test):
    
    # Label Encoding
    encoder = LabelEncoder()
    y_train = encoder.fit_transform(y_train)
    y_test = encoder.transform(y_test)

    # Print for debugging purpose
    print(encoder.classes_)
    print(np.unique(y_train))
    print(np.unique(y_test))

    # Feature Scaling
    scaler = StandardScaler()
    X_train = scaler.fit_transform(X_train) # Learn the Meand and std and transform the data
    X_test = scaler.transform(X_test) # Transform the data using the learned mean and std

    return (
        X_train,
        X_test,
        y_train,
        y_test,
        scaler,
        encoder
    )  

In [5]:
def build_model():

    model = LogisticRegression(

        penalty=LR_PENALTY,
        C=LR_C,
        solver=LR_SOLVER,
        max_iter=LR_MAX_ITER,
        random_state=RANDOM_SEED

    )

    return model

In [6]:
def train_model(
        model,
        X_train,
        y_train
):

    model.fit(
        X_train,
        y_train
    )

    return model

In [7]:
def evaluate_model(
        model,
        X_test,
        y_test,
        encoder
):
    probabilities = model.predict_proba(X_test)[:, 1]

    predictions = (
        probabilities >= THRESHOLD
    ).astype(int)

    actual = y_test

    # Confusion Matrix
    cm = confusion_matrix(actual, predictions)

    # Compute metrics
    metrics_dictionary = {
        "Accuracy": accuracy_score(actual, predictions),
        "Precision": precision_score(actual, predictions),
        "Recall": recall_score(actual, predictions),
        "F1 Score": f1_score(actual, predictions)
    }

    report = classification_report(
        actual,
        predictions,
        target_names=encoder.classes_,
        output_dict=True
    )

    print(classification_report(
        actual,
        predictions,
        target_names=encoder.classes_
    ))

    return {
        "metrics": metrics_dictionary,
        "predictions": predictions,
        "probabilities": probabilities,
        "actual": actual,
        "confusion_matrix": cm,
        "classification_report": report
    }

In [8]:
''' The following are saved:
    Model weights (logistic Regressoin.pt)
    Scaler (scaler.pkl)
    Encoder (encoder.pkl)
    Metrics (metrics.csv)
    Confusion Matrix (confusion_matrix.csv)
    Classification Report (classification_report.csv) '''
    
def save_results(
    participant,
    model,
    scaler,
    encoder,
    evaluation
):
    
    metrics = evaluation["metrics"]
    predictions = evaluation["predictions"]
    probabilities = evaluation["probabilities"]
    actual = evaluation["actual"]
    cm = evaluation["confusion_matrix"]
    report = evaluation["classification_report"]
    
    # Save Model
    participant_output = Path(OUTPUT_ROOT, participant)
    os.makedirs(participant_output, exist_ok=True)
    joblib.dump(model, Path(participant_output, "logistic_regression.pkl"))
    
    # Save Scaler
    joblib.dump(scaler,Path(participant_output, "scaler.pkl"))
    
    # Save Encoder
    joblib.dump(encoder,Path(participant_output, "encoder.pkl"))
    
    # Save Metrics
    metrics_df = pd.DataFrame([metrics])

    metrics_df.to_csv(Path(participant_output, "metrics.csv"),index=False)
        
    # Save Confusion Matrix
    cm_df = pd.DataFrame(cm,index=encoder.classes_,columns=encoder.classes_)
    cm_df.to_csv(Path(participant_output, "confusion_matrix.csv"))
    
    # Save classification report
    report_df = pd.DataFrame(report).transpose()
    report_df.to_csv(Path(participant_output, "classification_report.csv"))
    
    # Save predictions
    prediction_df = pd.DataFrame({
        "Actual": actual,
        "Prediction": predictions,
        "Probability": probabilities
    })

    prediction_df.to_csv(Path(participant_output, "predictions.csv"),index=False)

#### RUN MODEL

In [9]:
summary_results = []

participants = sorted(os.listdir(FEATURE_ROOT))

for participant in participants:

    print(f"Training {participant}")
    
    # Load participant data
    X, y = load_participant_data(participant)
    
    # Load data
    skf = StratifiedKFold(
        n_splits=5,
        shuffle=True,
        random_state=RANDOM_SEED
    )

    # Store metrics for each fold for this participant
    participant_metrics = []
    
    for fold, (train_idx, test_idx) in enumerate(skf.split(X, y), start=1):

        print(f"\nFold {fold}/5")

        X_train = X.iloc[train_idx]
        X_test = X.iloc[test_idx]

        y_train = y.iloc[train_idx]
        y_test = y.iloc[test_idx]
        
        # 20% Test, 80% (Train + Validation)
        X_train, X_val, y_train, y_val = train_test_split(
            X_train,
            y_train,
            test_size=0.20,
            stratify=y_train,
            random_state=RANDOM_SEED
        )
    
        # Preprocess data
        X_train, X_test, y_train, y_test, scaler, encoder = preprocess_data(X_train, X_test, y_train, y_test)

        # Build model
        model = build_model()

        # Train model and return training history
        model = train_model(model, X_train, y_train)

        # Evaluate model and return evaluation metrics
        evaluation = evaluate_model(model, X_test, y_test, encoder)

        participant_metrics.append(evaluation["metrics"])

        # Save results
        save_results(
            f"{participant}/fold_{fold}",
            model,
            scaler,
            encoder,
            evaluation
        )
    
    # After all folds are done, compute mean and std of metrics for this participant
    metrics_df = pd.DataFrame(participant_metrics)

    # Save summary results for this participant
    summary_results.append({
        "Participant": participant,

        "Accuracy Mean": metrics_df["Accuracy"].mean(),
        "Accuracy Std": metrics_df["Accuracy"].std(),

        "Precision Mean": metrics_df["Precision"].mean(),
        "Precision Std": metrics_df["Precision"].std(),

        "Recall Mean": metrics_df["Recall"].mean(),
        "Recall Std": metrics_df["Recall"].std(),

        "F1 Mean": metrics_df["F1 Score"].mean(),
        "F1 Std": metrics_df["F1 Score"].std()
    })

# Save summary results as CSV
summary_df = pd.DataFrame(summary_results)
summary_df.to_csv(
    Path(OUTPUT_ROOT,"summary_results.csv"),
    index=False
)
    

Training p11
p11: 9 CSV files
Total samples: 811
Index(['F0semitoneFrom27.5Hz_sma3nz_amean',
       'F0semitoneFrom27.5Hz_sma3nz_stddevNorm',
       'F0semitoneFrom27.5Hz_sma3nz_percentile20.0',
       'F0semitoneFrom27.5Hz_sma3nz_percentile50.0',
       'F0semitoneFrom27.5Hz_sma3nz_percentile80.0',
       'F0semitoneFrom27.5Hz_sma3nz_pctlrange0-2',
       'F0semitoneFrom27.5Hz_sma3nz_meanRisingSlope',
       'F0semitoneFrom27.5Hz_sma3nz_stddevRisingSlope',
       'F0semitoneFrom27.5Hz_sma3nz_meanFallingSlope',
       'F0semitoneFrom27.5Hz_sma3nz_stddevFallingSlope', 'loudness_sma3_amean',
       'loudness_sma3_stddevNorm', 'loudness_sma3_percentile20.0',
       'loudness_sma3_percentile50.0', 'loudness_sma3_percentile80.0',
       'loudness_sma3_pctlrange0-2', 'loudness_sma3_meanRisingSlope',
       'loudness_sma3_stddevRisingSlope', 'loudness_sma3_meanFallingSlope',
       'loudness_sma3_stddevFallingSlope', 'spectralFlux_sma3_amean',
       'spectralFlux_sma3_stddevNorm', 'mfcc1_sma

/home/interactionlab/anaconda3/envs/model_env/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:1403: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(
/home/interactionlab/anaconda3/envs/model_env/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:1403: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.wa

              precision    recall  f1-score   support

  disengaged       0.62      0.57      0.59        81
     engaged       0.60      0.65      0.63        81

    accuracy                           0.61       162
   macro avg       0.61      0.61      0.61       162
weighted avg       0.61      0.61      0.61       162


Fold 4/5
['disengaged' 'engaged']
[0 1]
[0 1]
              precision    recall  f1-score   support

  disengaged       0.66      0.63      0.65        82
     engaged       0.64      0.66      0.65        80

    accuracy                           0.65       162
   macro avg       0.65      0.65      0.65       162
weighted avg       0.65      0.65      0.65       162


Fold 5/5
['disengaged' 'engaged']
[0 1]
[0 1]
              precision    recall  f1-score   support

  disengaged       0.57      0.60      0.58        82
     engaged       0.57      0.54      0.55        80

    accuracy                           0.57       162
   macro avg       0.57      0.57 

/home/interactionlab/anaconda3/envs/model_env/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:1403: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(
/home/interactionlab/anaconda3/envs/model_env/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:1403: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.wa

              precision    recall  f1-score   support

  disengaged       0.58      0.53      0.55        93
     engaged       0.56      0.62      0.59        92

    accuracy                           0.57       185
   macro avg       0.57      0.57      0.57       185
weighted avg       0.57      0.57      0.57       185


Fold 4/5
['disengaged' 'engaged']
[0 1]
[0 1]
              precision    recall  f1-score   support

  disengaged       0.55      0.65      0.59        93
     engaged       0.57      0.47      0.51        92

    accuracy                           0.56       185
   macro avg       0.56      0.56      0.55       185
weighted avg       0.56      0.56      0.55       185


Fold 5/5
['disengaged' 'engaged']
[0 1]
[0 1]
              precision    recall  f1-score   support

  disengaged       0.53      0.59      0.56        93
     engaged       0.54      0.48      0.51        92

    accuracy                           0.54       185
   macro avg       0.54      0.53 

...

/home/interactionlab/anaconda3/envs/model_env/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:1403: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(
/home/interactionlab/anaconda3/envs/model_env/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:1403: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.wa

              precision    recall  f1-score   support

  disengaged       0.62      0.58      0.60        92
     engaged       0.59      0.63      0.61        90

    accuracy                           0.60       182
   macro avg       0.61      0.60      0.60       182
weighted avg       0.61      0.60      0.60       182


Fold 5/5
['disengaged' 'engaged']
[0 1]
[0 1]
              precision    recall  f1-score   support

  disengaged       0.62      0.63      0.62        92
     engaged       0.61      0.60      0.61        90

    accuracy                           0.62       182
   macro avg       0.62      0.62      0.62       182
weighted avg       0.62      0.62      0.62       182

Training p5
p5: 5 CSV files
Total samples: 354
Index(['F0semitoneFrom27.5Hz_sma3nz_amean',
       'F0semitoneFrom27.5Hz_sma3nz_stddevNorm',
       'F0semitoneFrom27.5Hz_sma3nz_percentile20.0',
       'F0semitoneFrom27.5Hz_sma3nz_percentile50.0',
       'F0semitoneFrom27.5Hz_sma3nz_percentile80.0',


/home/interactionlab/anaconda3/envs/model_env/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:1403: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(
/home/interactionlab/anaconda3/envs/model_env/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:1403: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.wa

Training p7
p7: 8 CSV files
Total samples: 479
Index(['F0semitoneFrom27.5Hz_sma3nz_amean',
       'F0semitoneFrom27.5Hz_sma3nz_stddevNorm',
       'F0semitoneFrom27.5Hz_sma3nz_percentile20.0',
       'F0semitoneFrom27.5Hz_sma3nz_percentile50.0',
       'F0semitoneFrom27.5Hz_sma3nz_percentile80.0',
       'F0semitoneFrom27.5Hz_sma3nz_pctlrange0-2',
       'F0semitoneFrom27.5Hz_sma3nz_meanRisingSlope',
       'F0semitoneFrom27.5Hz_sma3nz_stddevRisingSlope',
       'F0semitoneFrom27.5Hz_sma3nz_meanFallingSlope',
       'F0semitoneFrom27.5Hz_sma3nz_stddevFallingSlope', 'loudness_sma3_amean',
       'loudness_sma3_stddevNorm', 'loudness_sma3_percentile20.0',
       'loudness_sma3_percentile50.0', 'loudness_sma3_percentile80.0',
       'loudness_sma3_pctlrange0-2', 'loudness_sma3_meanRisingSlope',
       'loudness_sma3_stddevRisingSlope', 'loudness_sma3_meanFallingSlope',
       'loudness_sma3_stddevFallingSlope', 'spectralFlux_sma3_amean',
       'spectralFlux_sma3_stddevNorm', 'mfcc1_sma3_

/home/interactionlab/anaconda3/envs/model_env/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:1403: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(
/home/interactionlab/anaconda3/envs/model_env/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:1403: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.wa